# AlphaEarth Embeddings -> Landslide Probability U-Net

Run these cells top to bottom. The only manual step is downloading the Earth Engine export to the `ALPHAEARTH_RAW_PATH` shown below.

In [1]:
from pathlib import Path
import json
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = Path('/home/abdullah/fire-debrisflow-ml')
OUT_DIR = Path('/mnt/c/Users/amehedi/Downloads/pioneer/output')

AOI_PATH = OUT_DIR / 'huc_pioneer1.shp'
TEMPLATE_PATH = OUT_DIR / 'topographic__elevation.tif'  # 30 m grid template
TARGET_PATH = OUT_DIR / 'landslide__probability_of_failure.tif'

ALPHAEARTH_RAW_PATH = OUT_DIR / 'alphaearth_2023_pioneer_raw.tif'
ALPHAEARTH_ALIGNED_PATH = OUT_DIR / 'alphaearth_2023_pioneer_aligned_30m.tif'
MODEL_OUT_PATH = OUT_DIR / 'alphaearth_unet_landslide_probability.pt'

if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from deep.unet import UNetRegressor

ALPHAEARTH_BANDS = [f'A{i:02d}' for i in range(64)]
print('AOI:', AOI_PATH)
print('Template:', TEMPLATE_PATH)
print('Target:', TARGET_PATH)
print('Raw AlphaEarth:', ALPHAEARTH_RAW_PATH)

AOI: /mnt/c/Users/amehedi/Downloads/pioneer/output/huc_pioneer1.shp
Template: /mnt/c/Users/amehedi/Downloads/pioneer/output/topographic__elevation.tif
Target: /mnt/c/Users/amehedi/Downloads/pioneer/output/landslide__probability_of_failure.tif
Raw AlphaEarth: /mnt/c/Users/amehedi/Downloads/pioneer/output/alphaearth_2023_pioneer_raw.tif


## 1. Export AlphaEarth From Earth Engine

Run the next cell once. When the task finishes in Earth Engine / Google Drive, download the GeoTIFF and place it here:

`/mnt/c/Users/amehedi/Downloads/pioneer/output/alphaearth_2023_pioneer_raw.tif`

In [2]:
def start_alphaearth_drive_export(year=2023, export_scale=30, drive_folder='earthengine'):
    import ee

    try:
        ee.Initialize()
    except Exception:
        ee.Authenticate()
        ee.Initialize()

    aoi = gpd.read_file(AOI_PATH).to_crs(4326)
    geom = aoi.geometry.union_all() if hasattr(aoi.geometry, 'union_all') else aoi.geometry.unary_union
    geojson = json.loads(gpd.GeoSeries([geom], crs='EPSG:4326').to_json())['features'][0]['geometry']
    aoi_ee = ee.Geometry(geojson)

    with rasterio.open(TEMPLATE_PATH) as template:
        target_crs = template.crs.to_string()

    image = (
        ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
        .filterDate(f'{year}-01-01', f'{year + 1}-01-01')
        .filterBounds(aoi_ee)
        .mosaic()
        .select(ALPHAEARTH_BANDS)
        .clip(aoi_ee)
        .toFloat()
    )

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=f'alphaearth_{year}_pioneer_raw',
        folder=drive_folder,
        fileNamePrefix=f'alphaearth_{year}_pioneer_raw',
        region=aoi_ee,
        crs=target_crs,
        scale=export_scale,
        maxPixels=1e13,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True},
    )
    task.start()
    print('Started export task:', task.id)
    return task

# Uncomment this line to start the export.
# task = start_alphaearth_drive_export(year=2023, export_scale=30)


## 2. Save Landlab Probability Target

If `landslide__probability_of_failure.tif` does not exist yet, run this cell in the same notebook/session where your Landlab `grid` object exists after `LS_prob.calculate_landslide_probability()`.

In [3]:
def write_landlab_probability_target_from_grid(grid):
    field = 'landslide__probability_of_failure'
    if field not in grid.at_node:
        raise KeyError(f'{field} not found in grid.at_node')

    with rasterio.open(TEMPLATE_PATH) as template:
        template_arr = template.read(1)
        profile = template.profile.copy()
        height, width = template.height, template.width
        template_nodata = template.nodata

    values = np.asarray(grid.at_node[field], dtype='float32')
    if values.size != height * width:
        raise ValueError(f'Grid has {values.size} nodes, template has {height * width} pixels')

    nodata = -9999.0
    target = values.reshape(height, width).astype('float32')
    invalid = ~np.isfinite(template_arr)
    if template_nodata is not None:
        invalid |= np.isclose(template_arr, template_nodata)
    target[invalid] = nodata

    profile.update(driver='GTiff', dtype='float32', count=1, nodata=nodata, compress='lzw')
    with rasterio.open(TARGET_PATH, 'w', **profile) as dst:
        dst.write(target, 1)
    print('Wrote:', TARGET_PATH)

if TARGET_PATH.exists():
    print('Target exists:', TARGET_PATH)
elif 'grid' in globals():
    write_landlab_probability_target_from_grid(grid)
else:
    print('Target not found. Run Landlab first, then call write_landlab_probability_target_from_grid(grid).')


Target not found. Run Landlab first, then call write_landlab_probability_target_from_grid(grid).


## 3. Align AlphaEarth To The 30 m Template

This converts the raw AlphaEarth GeoTIFF to exactly the same CRS, transform, shape, and cell grid as `topographic__elevation.tif`.

In [4]:
def align_alphaearth_to_template():
    if not ALPHAEARTH_RAW_PATH.exists():
        raise FileNotFoundError(f'Missing raw AlphaEarth file: {ALPHAEARTH_RAW_PATH}')

    nodata = -9999.0
    with rasterio.open(TEMPLATE_PATH) as template, rasterio.open(ALPHAEARTH_RAW_PATH) as src:
        template_arr = template.read(1)
        template_invalid = ~np.isfinite(template_arr)
        if template.nodata is not None:
            template_invalid |= np.isclose(template_arr, template.nodata)

        profile = template.profile.copy()
        profile.update(driver='GTiff', dtype='float32', count=src.count, nodata=nodata, compress='lzw')

        with rasterio.open(ALPHAEARTH_ALIGNED_PATH, 'w', **profile) as dst:
            for band in range(1, src.count + 1):
                out = np.full((template.height, template.width), nodata, dtype='float32')
                reproject(
                    source=rasterio.band(src, band),
                    destination=out,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    src_nodata=src.nodata,
                    dst_transform=template.transform,
                    dst_crs=template.crs,
                    dst_nodata=nodata,
                    resampling=Resampling.average,
                )
                out[template_invalid] = nodata
                dst.write(out, band)

    print('Wrote:', ALPHAEARTH_ALIGNED_PATH)

if ALPHAEARTH_ALIGNED_PATH.exists():
    print('Aligned AlphaEarth exists:', ALPHAEARTH_ALIGNED_PATH)
else:
    align_alphaearth_to_template()


FileNotFoundError: Missing raw AlphaEarth file: /mnt/c/Users/amehedi/Downloads/pioneer/output/alphaearth_2023_pioneer_raw.tif

## 4. Load Data

In [5]:
def read_single_band(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype('float32')
        nodata = src.nodata
    invalid = ~np.isfinite(arr)
    if nodata is not None:
        invalid |= np.isclose(arr, nodata)
    return arr, invalid

def read_multiband(path):
    with rasterio.open(path) as src:
        arr = src.read().astype('float32')
        nodata = src.nodata
    invalid = ~np.all(np.isfinite(arr), axis=0)
    if nodata is not None:
        invalid |= np.any(np.isclose(arr, nodata), axis=0)
    return arr, invalid

X, emb_invalid = read_multiband(ALPHAEARTH_ALIGNED_PATH)
y, target_invalid = read_single_band(TARGET_PATH)
_, template_invalid = read_single_band(TEMPLATE_PATH)

y = np.clip(y, 0.0, 1.0).astype('float32')
valid_mask = ~(emb_invalid | target_invalid | template_invalid)

print('X shape [bands, rows, cols]:', X.shape)
print('y shape:', y.shape)
print('valid pixels:', int(valid_mask.sum()), 'of', valid_mask.size)

if X.shape[1:] != y.shape:
    raise ValueError(f'Shape mismatch: {X.shape[1:]} vs {y.shape}')
if valid_mask.sum() == 0:
    raise ValueError('No valid pixels remain after masking')

RasterioIOError: /mnt/c/Users/amehedi/Downloads/pioneer/output/alphaearth_2023_pioneer_aligned_30m.tif: No such file or directory

## 5. Patch Split And Normalization

In [ ]:
PATCH_SIZE = 128
STRIDE = 64
BLOCK_SIZE = 256
TRAIN_FRACTION = 0.8
MIN_VALID_FRACTION = 0.50
SEED = 42

def starts(start, stop, patch_size, stride):
    if stop - start < patch_size:
        return []
    s = list(range(start, stop - patch_size + 1, stride))
    last = stop - patch_size
    if not s or s[-1] != last:
        s.append(last)
    return s

def make_split(valid):
    rng = np.random.default_rng(SEED)
    h, w = valid.shape
    blocks = [
        (r, min(r + BLOCK_SIZE, h), c, min(c + BLOCK_SIZE, w))
        for r in range(0, h, BLOCK_SIZE)
        for c in range(0, w, BLOCK_SIZE)
    ]
    blocks = [b for b in blocks if valid[b[0]:b[1], b[2]:b[3]].any()]
    if len(blocks) < 2:
        raise ValueError('Need at least two valid blocks')
    rng.shuffle(blocks)
    n_train = min(max(int(round(len(blocks) * TRAIN_FRACTION)), 1), len(blocks) - 1)
    train_blocks, test_blocks = blocks[:n_train], blocks[n_train:]

    def origins(block_list):
        out = []
        for r0, r1, c0, c1 in block_list:
            for r in starts(r0, r1, PATCH_SIZE, STRIDE):
                for c in starts(c0, c1, PATCH_SIZE, STRIDE):
                    m = valid[r:r + PATCH_SIZE, c:c + PATCH_SIZE]
                    if float(m.mean()) >= MIN_VALID_FRACTION:
                        out.append((r, c))
        return out

    return train_blocks, test_blocks, origins(train_blocks), origins(test_blocks)

train_blocks, test_blocks, train_origins, test_origins = make_split(valid_mask)
if not train_origins or not test_origins:
    raise ValueError('No train/test patches. Lower MIN_VALID_FRACTION or PATCH_SIZE.')

train_region = np.zeros_like(valid_mask, dtype=bool)
for r0, r1, c0, c1 in train_blocks:
    train_region[r0:r1, c0:c1] = True
norm_mask = train_region & valid_mask

means = np.array([X[i][norm_mask].mean() for i in range(X.shape[0])], dtype='float32')
stds = np.array([X[i][norm_mask].std() for i in range(X.shape[0])], dtype='float32')
stds[stds < 1e-6] = 1.0

Xn = ((X - means[:, None, None]) / stds[:, None, None]).astype('float32')
Xn = np.clip(np.nan_to_num(Xn, nan=0.0, posinf=0.0, neginf=0.0), -20.0, 20.0)
Xn[:, ~valid_mask] = 0.0
y_clean = np.where(valid_mask, y, 0.0).astype('float32')

print('train patches:', len(train_origins))
print('test patches:', len(test_origins))


## 6. Train Simple U-Net And Report R2

In [ ]:
class RasterPatchDataset(Dataset):
    def __init__(self, x, y, valid, origins):
        self.x = x
        self.y = y
        self.valid = valid
        self.origins = origins

    def __len__(self):
        return len(self.origins)

    def __getitem__(self, idx):
        r, c = self.origins[idx]
        ps = PATCH_SIZE
        xb = self.x[:, r:r + ps, c:c + ps]
        yb = self.y[r:r + ps, c:c + ps][None, :, :]
        mb = self.valid[r:r + ps, c:c + ps][None, :, :].astype('float32')
        return torch.from_numpy(xb), torch.from_numpy(yb), torch.from_numpy(mb)

def masked_mse(pred, target, mask):
    return (((pred - target) ** 2) * mask).sum() / mask.sum().clamp_min(1.0)

@torch.no_grad()
def r2_for_loader(model, loader, device):
    model.eval()
    preds, truths = [], []
    for xb, yb, mb in loader:
        pred = model(xb.to(device)).cpu().numpy().reshape(-1)
        yy = yb.numpy().reshape(-1)
        mm = mb.numpy().reshape(-1) > 0.5
        preds.append(pred[mm])
        truths.append(yy[mm])
    pred = np.concatenate(preds)
    true = np.concatenate(truths)
    ss_res = np.sum((true - pred) ** 2)
    ss_tot = np.sum((true - true.mean()) ** 2)
    return float(1.0 - ss_res / ss_tot), float(np.sqrt(np.mean((true - pred) ** 2)))

torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_ds = RasterPatchDataset(Xn, y_clean, valid_mask, train_origins)
test_ds = RasterPatchDataset(Xn, y_clean, valid_mask, test_origins)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
train_eval_loader = DataLoader(train_ds, batch_size=4, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False)

model = nn.Sequential(
    UNetRegressor(in_channels=Xn.shape[0], base_channels=16),
    nn.Sigmoid(),
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

EPOCHS = 25
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for xb, yb, mb in train_loader:
        xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = masked_mse(model(xb), yb, mb)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.item()))

    train_r2, train_rmse = r2_for_loader(model, train_eval_loader, device)
    test_r2, test_rmse = r2_for_loader(model, test_loader, device)
    history.append({'epoch': epoch, 'train_r2': train_r2, 'test_r2': test_r2, 'test_rmse': test_rmse})
    print(f'epoch {epoch:02d} loss={np.mean(losses):.5f} train_r2={train_r2:.3f} test_r2={test_r2:.3f} test_rmse={test_rmse:.4f}')

torch.save({
    'model_state_dict': model.state_dict(),
    'in_channels': int(Xn.shape[0]),
    'base_channels': 16,
    'norm_mean': means.tolist(),
    'norm_std': stds.tolist(),
    'history': history,
}, MODEL_OUT_PATH)

print('saved model:', MODEL_OUT_PATH)
print('final train R2:', history[-1]['train_r2'])
print('final test R2:', history[-1]['test_r2'])


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot([h['epoch'] for h in history], [h['train_r2'] for h in history], label='train R2')
plt.plot([h['epoch'] for h in history], [h['test_r2'] for h in history], label='test R2')
plt.axhline(0, color='k', linewidth=0.8)
plt.xlabel('Epoch')
plt.ylabel('R2')
plt.legend()
plt.tight_layout()
plt.show()